# 05 — Entity-Level Validation

Select/inspect thresholds using the challenge-style macro F0.5 calculation.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import fbeta_score
ROOT=Path.cwd().resolve()
if not (ROOT/'data').exists() and (ROOT.parent/'data').exists(): ROOT=ROOT.parent
features=pd.read_pickle(ROOT/'output'/'training_features.pkl')
feature_cols=['name_similarity','name_token_similarity','address_similarity','address_token_similarity','name_exact','country_match','name_missing','address_missing','name_length_diff','address_length_diff']
model=joblib.load(ROOT/'models'/'entity_resolution_random_forest.joblib')
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
_,va=next(gss.split(features,features['label'],groups=features['source1_entity_id']))
val=features.iloc[va].copy(); val['prob']=model.predict_proba(val[feature_cols])[:,1]
TRAIN=ROOT/'data'/'train'; gt=pd.read_csv(TRAIN/'train_ground_truth.tsv',sep='\t')
truth={r.source1_entity_id:(set(str(r.matched_entity_ids).split(',')) if pd.notna(r.matched_entity_ids) and str(r.matched_entity_ids).strip() else set()) for r in gt.itertuples(index=False)}

In [ ]:
def macro_entity_f05(df,threshold):
    vals=[]
    for s1id, grp in df.groupby('source1_entity_id'):
        pred=set(grp.loc[grp.prob>=threshold,'candidate_entity_id'])
        true=truth.get(s1id,set())
        tp=len(pred&true); fp=len(pred-true); fn=len(true-pred)
        precision=tp/(tp+fp) if tp+fp else 1.0
        recall=tp/(tp+fn) if tp+fn else 1.0
        f=(1.25*precision*recall)/(0.25*precision+recall) if precision+recall else 0.0
        vals.append(f)
    return float(np.mean(vals))

results=[]
for t in np.arange(0.30,0.951,0.05):
    results.append((round(float(t),2),macro_entity_f05(val,t)))
res=pd.DataFrame(results,columns=['threshold','macro_f0_5']).sort_values('macro_f0_5',ascending=False)
display(res)
best=float(res.iloc[0].threshold)
(ROOT/'models'/'threshold.txt').write_text(str(best))
print('Selected validation threshold:',best)